# SDTM AE Domain Transformation

---
## Overview
Notebook 5 of 8.

This notebook transforms raw adverse event data into SDTM Adverse Events (AE) domain format.

Run the entire notebook to load the data.

---

## Source Data

- **Source:** bronze.ae_raw
- **Domain:** Adverse Events (AE)

---

## Transformation Steps

**1. Data Quality**  
Loads bronze data and transforms records to domain standard

**2. SDTM Mapping**  
Maps source columns to SDTM variables:
- Event identifiers (AESEQ, AESPID)
- Event details (AETERM, AEDECOD)
- Severity and relationship (AESEV, AEREL)
- Dates and outcomes (AESTDTC, AEENDTC, AEOUT)

**3. Controlled Terminology**  

- Severity grading (Serious Flag)
- Outcomes

---

## Target

SDTM-compliant data written to **silver.sdtm_ae** Delta table.

---

In [0]:
from pyspark.sql.functions import udf, col, lit, when
from pyspark.sql.types import StringType

In [0]:
#global variables
CATALOG = "cluepoints_db_demo"
SCHEMA = "silver"

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}") #should exist if demog run first

DataFrame[]

In [0]:
#decode 'serious' flag
#see previous notebook comments about replacing UDFs with Scala

@udf(returnType=StringType())
def decode_aeser(aeser_code):
    if aeser_code == 0:
        return 'N'
    elif aeser_code == 1:
        return 'Y'
    return None

In [0]:
# udf to decode outcomes - this is an assumed mapping, needs verification - there's no code 4 in the source data
# https://www.cdisc.org/kb/examples/adverse-events-ae-device-event-reporting-acrf-73161712

@udf(returnType=StringType())
def decode_aeout(aeout_code):
    if aeout_code == 1:
        return 'RECOVERED/RESOLVED'
    elif aeout_code == 2:
        return 'RECOVERING/RESOLVING'
    elif aeout_code == 3:
        return 'NOT RECOVERED/NOT RESOLVED'
    elif aeout_code == 5:
        return 'FATAL'
    elif aeout_code == 6:
        return 'UNKNOWN'
    return None

In [0]:

ae = spark.table("cluepoints_db_demo.bronze.ae")

# Create spark dataframe - alias the columns and add the decode UDFs
sdtm_ae = ae.select(
    col('subjid').alias('SUBJID'),
    col('center').cast('string').alias('SITEID'), #no decode for site id, in real world this will be foreign key to a site table
    col('hlgtn').alias('AEHLGT'),
    col('aestrda').alias('AESTDTC'),
    decode_aeser(col('aeseryn')).alias('AESER'),
    decode_aeout(col('aeout_n')).alias('AEOUT'),
    col('source_file'),
    col('loaded_at'),
    col('ingestion_id')
)


In [0]:
# Write to Delta - overwrite all target data
sdtm_ae.write.format('delta').mode('overwrite').saveAsTable('cluepoints_db_demo.silver.ae_sdtm')

In [0]:
#display the table
display(spark.sql("select * from cluepoints_db_demo.silver.ae_sdtm limit 10"))

SUBJID,SITEID,AEHLGT,AESTDTC,AESER,AEOUT,source_file,loaded_at,ingestion_id
840414002,414,null,2004-09-30,N,RECOVERED/RESOLVED,ae.csv,2026-01-04T13:51:42.108Z,20260104_135139
826290005,290,GENERAL SYSTEM DISORDERS NEC,null,N,NOT RECOVERED/NOT RESOLVED,ae.csv,2026-01-04T13:51:42.108Z,20260104_135139
826290002,290,null,null,N,RECOVERED/RESOLVED,ae.csv,2026-01-04T13:51:42.108Z,20260104_135139
616187066,188,ANXIETY DISORDERS AND SYMPTOMS,null,N,RECOVERED/RESOLVED,ae.csv,2026-01-04T13:51:42.108Z,20260104_135139
616187061,189,HEADACHES,null,N,RECOVERED/RESOLVED,ae.csv,2026-01-04T13:51:42.108Z,20260104_135139
616187053,188,CARDIAC ARRHYTHMIAS,null,N,RECOVERED/RESOLVED,ae.csv,2026-01-04T13:51:42.108Z,20260104_135139
616187050,188,INFECTIONS - PATHOGEN CLASS UNSPECIFIED,null,N,RECOVERED/RESOLVED,ae.csv,2026-01-04T13:51:42.108Z,20260104_135139
616187049,189,CARDIAC ARRHYTHMIAS,null,Y,RECOVERED/RESOLVED,ae.csv,2026-01-04T13:51:42.108Z,20260104_135139
616187048,187,CARDIAC ARRHYTHMIAS,null,N,RECOVERED/RESOLVED,ae.csv,2026-01-04T13:51:42.108Z,20260104_135139
616187043,189,CARDIAC ARRHYTHMIAS,null,N,RECOVERED/RESOLVED,ae.csv,2026-01-04T13:51:42.108Z,20260104_135139
